<a href="https://colab.research.google.com/github/mffg1993/KnotFigures/blob/main/TheKnotbookAppendix2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Plan #1:** Parallelizing the search for singularities

## Knotted field Calculation

In [2]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px


class OpticalKnot:
    """
    Tutorial-friendly class for constructing and visualizing
    braid-based optical knots.

    Parameters
    ----------
    s : int
        Number of strands.
    r : int
        Braid winding parameter.
    ell : int
        Lemniscate / Lissajous parameter.
    a, b : float
        Stretching factors in the transverse plane.
    """

    def __init__(self, s, r, ell, a=1.0, b=1.0):
        self.s = s
        self.r = r
        self.ell = ell
        self.a = a
        self.b = b

        # symbolic variables
        self.rho = sp.symbols("rho", positive=True, real=True)
        self.phi = sp.symbols("phi", real=True)
        self.u, self.vv, self.vp = sp.symbols("u vv vp")

        # symbolic results
        self.expr = None
        self.field_expr = None

        # numerical field data
        self.x = None
        self.y = None
        self.X = None
        self.Y = None
        self.E = None

    # --------------------------------------------------
    # 1. Geometry
    # --------------------------------------------------
    def braid_data(self, npts=500):
        """
        Return the open braid strands in parametric form.
        """
        h = np.linspace(0.0, 2.0 * np.pi, npts)
        strands = []

        for j in range(1, self.s + 1):
            arg = (self.r * h + 2.0 * np.pi * (j - 1)) / self.s
            x = self.a * np.cos(arg)
            y = (self.b / self.ell) * np.sin(self.ell * arg)
            z = h
            strands.append((x, y, z))

        return h, strands

    def plot_braid(
        self,
        npts=500,
        braid_width=10,
        endpoint_size=5,
        show_endpoints=True,
        show_top_bottom=True,
        show_cylinder=True,
        cylinder_radius=None,
        cylinder_opacity=0.12,
        lemniscate_width=6,
        height=750,
        width=850,
        title=None,
    ):
        """
        Interactive Plotly plot of the open braid.
        """
        h, strands = self.braid_data(npts=npts)

        colors = [
            "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
            "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf"
        ]

        fig = go.Figure()
        all_x, all_y, all_z = [], [], []

        for j, (x, y, z) in enumerate(strands, start=1):
            color = colors[(j - 1) % len(colors)]

            fig.add_trace(
                go.Scatter3d(
                    x=x, y=y, z=z,
                    mode="lines",
                    line=dict(color=color, width=braid_width),
                    name=f"strand {j}",
                )
            )

            if show_endpoints:
                fig.add_trace(
                    go.Scatter3d(
                        x=[x[0], x[-1]],
                        y=[y[0], y[-1]],
                        z=[z[0], z[-1]],
                        mode="markers",
                        marker=dict(size=endpoint_size, color=color),
                        showlegend=False,
                    )
                )

            all_x.append(x)
            all_y.append(y)
            all_z.append(z)

        all_x = np.concatenate(all_x)
        all_y = np.concatenate(all_y)

        if show_top_bottom:
            t = np.linspace(0.0, 2.0 * np.pi, 1000)
            xL = self.a * np.cos(t)
            yL = (self.b / self.ell) * np.sin(self.ell * t)

            fig.add_trace(
                go.Scatter3d(
                    x=xL,
                    y=yL,
                    z=np.zeros_like(t),
                    mode="lines",
                    line=dict(color="black", width=lemniscate_width),
                    name="bottom lemniscate",
                )
            )

            fig.add_trace(
                go.Scatter3d(
                    x=xL,
                    y=yL,
                    z=np.full_like(t, 2.0 * np.pi),
                    mode="lines",
                    line=dict(color="black", width=lemniscate_width),
                    name="top lemniscate",
                )
            )

        if show_cylinder:
            if cylinder_radius is None:
                radial_extent = np.sqrt(all_x**2 + all_y**2).max()
                cylinder_radius = 1.15 * radial_extent

            theta = np.linspace(0.0, 2.0 * np.pi, 80)
            zz = np.linspace(0.0, 2.0 * np.pi, 80)
            Theta, ZZ = np.meshgrid(theta, zz)

            Xc = cylinder_radius * np.cos(Theta)
            Yc = cylinder_radius * np.sin(Theta)
            Zc = ZZ

            fig.add_trace(
                go.Surface(
                    x=Xc,
                    y=Yc,
                    z=Zc,
                    opacity=cylinder_opacity,
                    showscale=False,
                    colorscale=[[0, "lightgray"], [1, "lightgray"]],
                    hoverinfo="skip",
                    name="cylinder",
                )
            )

        if title is None:
            title = f"Open braid: s={self.s}, r={self.r}, ell={self.ell}"

        fig.update_layout(
            title=title,
            width=width,
            height=height,
            scene=dict(
                xaxis_title="x",
                yaxis_title="y",
                zaxis_title="h",
                aspectmode="data",
            ),
            margin=dict(l=0, r=0, b=0, t=45),
        )

        fig.show()

    def plot_on_torus(
        self,
        R=1.0,
        npts=600,
        torus_res_u=80,
        torus_res_v=50,
        line_width=10,
        torus_opacity=0.30,
        show_torus=True,
        title=None,
    ):
        """
        Interactive Plotly plot of the closed braid on a torus.
        """
        h, strands = self.braid_data(npts=npts)

        colors = [
            "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
            "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf"
        ]

        fig = go.Figure()

        if show_torus:
            u = np.linspace(0.0, 2.0 * np.pi, torus_res_u)
            v = np.linspace(0.0, 2.0 * np.pi, torus_res_v)
            U, V = np.meshgrid(u, v)

            Xt = (R + self.a * np.cos(V)) * np.cos(U)
            Yt = (R + self.a * np.cos(V)) * np.sin(U)
            Zt = self.b * np.sin(V)

            fig.add_trace(
                go.Surface(
                    x=Xt,
                    y=Yt,
                    z=Zt,
                    opacity=torus_opacity,
                    showscale=False,
                    colorscale=[[0, "lightgray"], [1, "lightgray"]],
                    hoverinfo="skip",
                    name="torus",
                )
            )

        for j, (x, y, z) in enumerate(strands, start=1):
            Xc = (R + x) * np.cos(z)
            Yc = (R + x) * np.sin(z)
            Zc = y

            color = colors[(j - 1) % len(colors)]

            fig.add_trace(
                go.Scatter3d(
                    x=Xc,
                    y=Yc,
                    z=Zc,
                    mode="lines",
                    line=dict(color=color, width=line_width),
                    name=f"strand {j}",
                )
            )

        if title is None:
            title = f"Closed braid on torus: s={self.s}, r={self.r}, ell={self.ell}"

        fig.update_layout(
            title=title,
            scene=dict(
                xaxis_title="x",
                yaxis_title="y",
                zaxis_title="z",
                aspectmode="data",
            ),
            margin=dict(l=0, r=0, b=0, t=45),
        )

        fig.show()

    # --------------------------------------------------
    # 2. Symbolic polynomial
    # --------------------------------------------------
    def Z(self, j):
        phase = 2 * sp.pi * (j - 1) / self.s

        T  = self.vv**sp.Rational(self.r, self.s) * sp.exp(sp.I * phase)
        Ti = self.vp**sp.Rational(self.r, self.s) * sp.exp(-sp.I * phase)

        return (
            sp.Rational(self.a, 2) * (T + Ti)
            + sp.Rational(self.b, 2 * self.ell) * (T**self.ell - Ti**self.ell)
        )

    def build_polynomial(self):
        """
        Construct the numerator of the braid polynomial after stereographic substitution.
        """
        p = sp.prod(self.u - self.Z(j) for j in range(1, self.s + 1))
        p = sp.expand(p)

        expr = p.subs({
            self.u: (self.rho**2 - 1) / (self.rho**2 + 1),
            self.vv: 2 * self.rho * sp.exp(sp.I * self.phi) / (self.rho**2 + 1),
            self.vp: 2 * self.rho * sp.exp(-sp.I * self.phi) / (self.rho**2 + 1),
        })

        expr = sp.together(expr)
        num, _ = sp.fraction(expr)
        self.expr = sp.expand(num)
        return self.expr

    def show_polynomial(self):
        """
        Display the symbolic polynomial. Builds it if needed.
        """
        if self.expr is None:
            self.build_polynomial()
        return self.expr

    def polynomial_latex(self):
        """
        Return the polynomial in LaTeX form.
        """
        if self.expr is None:
            self.build_polynomial()
        return sp.latex(self.expr)

    # --------------------------------------------------
    # 3. Field construction
    # --------------------------------------------------
    def make_field(self, N=500, xmax=2.0, w0=1.0):
        """
        Construct the field
            E(rho,phi) = exp(-rho^2 / w0^2) * expr(rho,phi)
        on a Cartesian grid.
        """
        if self.expr is None:
            self.build_polynomial()

        self.field_expr = sp.exp(-(self.rho**2) / w0**2) * self.expr
        f_num = sp.lambdify((self.rho, self.phi), self.field_expr, modules="numpy")

        self.x = np.linspace(-xmax, xmax, N)
        self.y = np.linspace(-xmax, xmax, N)
        self.X, self.Y = np.meshgrid(self.x, self.y)

        RHO = np.sqrt(self.X**2 + self.Y**2)
        PHI = np.arctan2(self.Y, self.X)

        self.E = f_num(RHO, PHI)
        return self.x, self.y, self.X, self.Y, self.E

    # --------------------------------------------------
    # 4. Plotting the field
    # --------------------------------------------------
    def plot_field(self, figsize=(12, 5)):
        """
        Plot intensity and phase side by side.
        """
        if self.E is None:
            raise ValueError("Field not built yet. Run make_field(...) first.")

        fig, ax = plt.subplots(1, 2, figsize=figsize)

        im0 = ax[0].imshow(
            np.abs(self.E)**2,
            extent=[self.x.min(), self.x.max(), self.y.min(), self.y.max()],
            origin="lower",
            cmap='gray'
        )
        ax[0].set_title("Intensity")
        ax[0].set_xlabel("x")
        ax[0].set_ylabel("y")


        im1 = ax[1].imshow(
            np.angle(self.E),
            extent=[self.x.min(), self.x.max(), self.y.min(), self.y.max()],
            origin="lower",
            cmap='hsv',
            vmin=-np.pi,
            vmax=np.pi
        )
        ax[1].set_title("Phase")
        ax[1].set_xlabel("x")
        ax[1].set_ylabel("y")

        plt.tight_layout()
        plt.show()

## **Singularity recollection - No bounding box**

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import plotly.express as px
import plotly.graph_objects as go


##############################################################################################
# Propagation operators
##############################################################################################

def propTF(u1, L, la, z):
    """
    Fresnel propagation using transfer function approach.
    This version preserves your original behavior.
    """
    M, nn = u1.shape
    dx = L / M

    fx = np.arange(-1 / (2 * dx), 1 / (2 * dx), 1 / L)
    Fx, Fy = np.meshgrid(fx, fx)

    H = np.exp(-1j * np.pi * 0.25 * la * z * (Fx**2 + Fy**2))
    U2 = H * np.fft.fftshift(np.fft.fft2(u1))
    u2 = np.fft.ifft2(np.fft.ifftshift(U2))
    return u2


##############################################################################################
# Basic ordering function
##############################################################################################

def SortPoints(A0):
    AR = np.array(A0, dtype=float, copy=True)
    col, row = AR.shape

    point = AR[:, 0]
    Ord = np.zeros((col, row - 2))

    Ocol, Orow = Ord.shape
    Ord[:, -1] = point

    for ii in range(Orow - 1):
        remove1 = np.argmin(np.abs(np.sum(np.transpose(AR) - np.tile(point, (row, 1)), axis=1)))

        Ord[:, ii] = point

        AR = np.squeeze(AR[:, np.where(np.sum(np.transpose(AR) - np.tile(point, (row, 1)), axis=1) != 0)])
        col, row = AR.shape
        kk = np.argmin(np.sum((np.transpose(AR) - np.tile(point, (row, 1)))**2, axis=1))

        point = AR[:, kk]

    return Ord


##############################################################################################
# Plot of the Points
##############################################################################################

def KnotPlot(Ord):
    fig1 = px.scatter_3d(x=Ord[0, :], y=Ord[1, :], z=Ord[2, :])
    fig1.update_traces(marker=dict(color='red', size=5))

    fig4 = px.line_3d(x=Ord[0, :], y=Ord[1, :], z=Ord[2, :])
    fig4.update_traces(line=dict(color='black', width=3))

    fig3 = go.Figure(data=fig1.data + fig4.data)
    fig3.update_layout(
        autosize=False,
        width=800,
        height=800,
    )
    fig3.show()


##############################################################################################
# Intersection utilities -- preserves old behavior
##############################################################################################

def find_intersections_old(A, B):
    """
    Original-style polyline intersection finder.
    Keeps the same behavior as your old code.
    """
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)

    if len(A) < 2 or len(B) < 2:
        return np.array([]), np.array([])

    amin = lambda x1, x2: np.where(x1 < x2, x1, x2)
    amax = lambda x1, x2: np.where(x1 > x2, x1, x2)
    aall = lambda abools: np.dstack(abools).all(axis=2)

    def slope(line):
        d = np.diff(line, axis=0)
        with np.errstate(divide='ignore', invalid='ignore'):
            return d[:, 1] / d[:, 0]

    x11, x21 = np.meshgrid(A[:-1, 0], B[:-1, 0])
    x12, x22 = np.meshgrid(A[1:, 0],  B[1:, 0])
    y11, y21 = np.meshgrid(A[:-1, 1], B[:-1, 1])
    y12, y22 = np.meshgrid(A[1:, 1],  B[1:, 1])

    m1, m2 = np.meshgrid(slope(A), slope(B))

    with np.errstate(divide='ignore', invalid='ignore'):
        m1inv = 1 / m1
        m2inv = 1 / m2
        yi = (m1 * (x21 - x11 - m2inv * y21) + y11) / (1 - m1 * m2inv)
        xi = (yi - y21) * m2inv + x21

    xconds = (
        amin(x11, x12) < xi, xi <= amax(x11, x12),
        amin(x21, x22) < xi, xi <= amax(x21, x22)
    )
    yconds = (
        amin(y11, y12) < yi, yi <= amax(y11, y12),
        amin(y21, y22) < yi, yi <= amax(y21, y22)
    )

    mask = aall(xconds) & aall(yconds) & np.isfinite(xi) & np.isfinite(yi)
    return xi[mask], yi[mask]


##############################################################################################
# Contour extraction
##############################################################################################

def _contour_segments_zero(Z):
    """
    Old behavior: contours in pixel/index coordinates.
    """
    fig, ax = plt.subplots()
    cs = ax.contour(Z, levels=[0])
    plt.close(fig)

    if len(cs.allsegs) == 0:
        return []
    return cs.allsegs[0]


##############################################################################################
# Singularity extraction
##############################################################################################

# --------------------------------------------------------------------------------------------
# Single Plane Singularity extractor
# --------------------------------------------------------------------------------------------

from joblib import Parallel, delayed

def _extract_singularities_one_plane(
    U,
    ii,
    forward=True,
    JJ=None,
    use_gaussian_filter=False,
    waist_pixels=None,
    roi_radius_factor=2.0,
    center=None,
    patch_radius=3,
    intensity_threshold=1e-4,
    use_normalized_intensity=True,
    local_stat="max",
    global_intensity_max=None,
):
    Ny, Nx = U.shape

    if center is None:
        xc = (Nx - 1) / 2
        yc = (Ny - 1) / 2
    else:
        xc, yc = center

    # intensity for this plane
    Iplane = np.abs(U) ** 2
    if use_normalized_intensity and global_intensity_max is not None and global_intensity_max > 0:
        Iplane = Iplane / global_intensity_max

    do_filter = use_gaussian_filter and (waist_pixels is not None)
    roi_radius = roi_radius_factor * waist_pixels if do_filter else None

    if JJ is None:
        real_segs = _contour_segments_zero(np.real(U))
        imag_segs = _contour_segments_zero(np.imag(U))
    else:
        real_segs = _contour_segments_zero(np.real(U + JJ))
        imag_segs = _contour_segments_zero(np.imag(U + JJ))

    xi_keep = []
    yi_keep = []

    for seg1 in real_segs:
        for seg2 in imag_segs:
            xinter, yinter = find_intersections_old(seg1, seg2)

            if xinter.size == 0:
                continue

            if not do_filter:
                xi_keep.extend(xinter.tolist())
                yi_keep.extend(yinter.tolist())
                continue

            for x0, y0 in zip(xinter, yinter):
                r = np.sqrt((x0 - xc)**2 + (y0 - yc)**2)
                outside_roi = r > roi_radius

                ix = int(round(x0))
                iy = int(round(y0))

                x1 = max(0, ix - patch_radius)
                x2 = min(Nx, ix + patch_radius + 1)
                y1 = max(0, iy - patch_radius)
                y2 = min(Ny, iy + patch_radius + 1)

                patch = Iplane[y1:y2, x1:x2]

                if patch.size == 0:
                    local_intensity = 0.0
                elif local_stat == "max":
                    local_intensity = np.max(patch)
                elif local_stat == "mean":
                    local_intensity = np.mean(patch)
                else:
                    raise ValueError("local_stat must be 'max' or 'mean'")

                if not (outside_roi and (local_intensity < intensity_threshold)):
                    xi_keep.append(x0)
                    yi_keep.append(y0)

    xi_keep = np.asarray(xi_keep, dtype=float)
    yi_keep = np.asarray(yi_keep, dtype=float)

    sign = 1 if forward else -1
    zi_keep = sign * ii * np.ones_like(xi_keep)

    return xi_keep, yi_keep, zi_keep


# --------------------------------------------------------------------------------------------
# Parallel singularity extractor
# --------------------------------------------------------------------------------------------

def _singular_from_stack_parallel(
    field_stack,
    forward=True,
    apply_boundary_mask=False,
    boundary_radius=70,
    use_gaussian_filter=False,
    waist_pixels=None,
    roi_radius_factor=2.0,
    center=None,
    patch_radius=3,
    intensity_threshold=1e-4,
    use_normalized_intensity=True,
    local_stat="max",
    n_jobs=-1,
    verbose=True,
):
    Fstack = np.asarray(field_stack)
    if Fstack.ndim != 3:
        raise ValueError("field_stack must have shape (Nz, Ny, Nx)")

    Nz, Ny, Nx = Fstack.shape

    if center is None:
        xc = (Nx - 1) / 2
        yc = (Ny - 1) / 2
    else:
        xc, yc = center

    if apply_boundary_mask:
        xx = np.arange(Nx)
        yy = np.arange(Ny)
        xf, yf = np.meshgrid(xx, yy)
        JJ = np.sqrt((xf - xc)**2 + (yf - yc)**2) > boundary_radius
    else:
        JJ = None

    global_intensity_max = np.max(np.abs(Fstack) ** 2) if use_normalized_intensity else None

    results = Parallel(n_jobs=n_jobs, prefer="processes", verbose=10 if verbose else 0)(
        delayed(_extract_singularities_one_plane)(
            U=Fstack[ii],
            ii=ii,
            forward=forward,
            JJ=JJ,
            use_gaussian_filter=use_gaussian_filter,
            waist_pixels=waist_pixels,
            roi_radius_factor=roi_radius_factor,
            center=center,
            patch_radius=patch_radius,
            intensity_threshold=intensity_threshold,
            use_normalized_intensity=use_normalized_intensity,
            local_stat=local_stat,
            global_intensity_max=global_intensity_max,
        )
        for ii in range(Nz)
    )

    X_list, Y_list, Z_list = [], [], []

    for xk, yk, zk in results:
        if xk.size:
            X_list.append(xk)
            Y_list.append(yk)
            Z_list.append(zk)

    if not X_list:
        return np.zeros((3, 0), dtype=float)

    return np.array([
        np.concatenate(X_list),
        np.concatenate(Y_list),
        np.concatenate(Z_list),
    ], dtype=float)



##############################################################################################
# Main propagation + knot extraction
##############################################################################################

def compute_knot_propagation_legacy_compatible(
    E0,
    L,
    wavelength=1.0,
    Z0=15.0,
    nz=50,
    do_backward=True,
    order_points=True,
    apply_forward_mask=True,
    boundary_radius=70,
    use_gaussian_filter=False,
    waist_pixels=None,
    roi_radius_factor=2.0,
    center=None,
    patch_radius=3,
    intensity_threshold=1e-4,
    use_normalized_intensity=True,
    local_stat="max",
    n_jobs=-1,
    show_progress=True,
    verbose=True,
):
    """
    Version that reproduces the behavior of your old code as closely as possible,
    with optional Gaussian-based filtering and parallel singularity extraction.

    Parameters
    ----------
    E0 : 2D complex array
        Input field.
    L : float
        Window size passed into propTF, equivalent to your old 2*maxx.
    wavelength : float
        Wavelength parameter passed to propTF.
    Z0 : float
        Total propagation distance.
    nz : int
        Number of propagation steps.
    """

    E0 = np.asarray(E0, dtype=complex)

    if E0.ndim != 2:
        raise ValueError("E0 must be a 2D field.")
    if nz <= 0:
        raise ValueError("nz must be > 0.")
    if wavelength <= 0:
        raise ValueError("wavelength must be > 0.")

    Z = np.linspace(0, Z0, nz)
    dz = np.abs(Z[0] - Z[1]) if nz > 1 else Z0

    if verbose:
        print("Starting legacy-compatible propagation...")
        print(f"Field shape: {E0.shape}")
        print(f"L = {L}")
        print(f"wavelength = {wavelength}")
        print(f"nz = {nz}")
        print(f"dz = {dz}")

    # ------------------------------------------------------------------
    # Forward propagation: old behavior
    # ------------------------------------------------------------------
    F = [E0.copy()]
    U = E0.copy()

    iterator = range(nz)
    if show_progress:
        iterator = tqdm(iterator, total=nz, desc="forward propagation", leave=True)

    for _ in iterator:
        U = propTF(U, L, wavelength, dz)
        F.append(U.copy())

    # ------------------------------------------------------------------
    # Backward propagation: old behavior
    # ------------------------------------------------------------------
    FB = []
    if do_backward:
        U = E0.copy()

        iterator = range(1, nz + 1)
        if show_progress:
            iterator = tqdm(iterator, total=nz, desc="backward propagation", leave=True)

        for _ in iterator:
            U = propTF(U, L, wavelength, -dz)
            FB.append(U.copy())

    # ------------------------------------------------------------------
    # Singularity extraction
    # ------------------------------------------------------------------
    if verbose:
        print("Extracting forward singularities...")

    H = _singular_from_stack_parallel(
        F,
        forward=True,
        apply_boundary_mask=apply_forward_mask,
        boundary_radius=boundary_radius,
        use_gaussian_filter=use_gaussian_filter,
        waist_pixels=waist_pixels,
        roi_radius_factor=roi_radius_factor,
        center=center,
        patch_radius=patch_radius,
        intensity_threshold=intensity_threshold,
        use_normalized_intensity=use_normalized_intensity,
        local_stat=local_stat,
        n_jobs=n_jobs,
        verbose=verbose,
    )

    if do_backward:
        if verbose:
            print("Extracting backward singularities...")

        H2 = _singular_from_stack_parallel(
            FB,
            forward=False,
            apply_boundary_mask=False,
            boundary_radius=boundary_radius,
            use_gaussian_filter=use_gaussian_filter,
            waist_pixels=waist_pixels,
            roi_radius_factor=roi_radius_factor,
            center=center,
            patch_radius=patch_radius,
            intensity_threshold=intensity_threshold,
            use_normalized_intensity=use_normalized_intensity,
            local_stat=local_stat,
            n_jobs=n_jobs,
            verbose=verbose,
        )
    else:
        H2 = np.zeros((3, 0), dtype=float)

    # ------------------------------------------------------------------
    # Combine and order
    # ------------------------------------------------------------------
    TotalPoints = np.concatenate((H, H2), axis=1)

    if order_points and TotalPoints.shape[1] > 2:
        if verbose:
            print("Ordering points...")
        ord_points = SortPoints(TotalPoints)
    else:
        ord_points = TotalPoints

    if verbose:
        print("Done.")
        print(f"Total points found: {TotalPoints.shape[1]}")

    return {
        "E0": E0,
        "L": L,
        "wavelength": wavelength,
        "Z0": Z0,
        "nz": nz,
        "dz": dz,
        "forward_fields": np.asarray(F, dtype=complex),
        "backward_fields": np.asarray(FB, dtype=complex) if do_backward else None,
        "singular_forward": H,
        "singular_backward": H2,
        "total_points": TotalPoints,
        "ordered_points": ord_points,
    }

In [4]:

# Creation of the Knot
knot = OpticalKnot(s=2, r=3, ell=1, a=1, b=1)


maxx=8
NN=512
w0_t = 1.6


A=knot.make_field(N=NN, xmax=maxx, w0=w0_t)

E=A[4]
dx = (2*maxx) / E.shape[0]
w0_pix = int(w0_t/ dx)


result = compute_knot_propagation_legacy_compatible(
    E0=E,
    L=2*maxx,
    wavelength=1,
    Z0=15,
    nz=50,
    do_backward=True,
    order_points=True,
    apply_forward_mask=True,
    boundary_radius=70,
    use_gaussian_filter=True,
    waist_pixels=51.2,
    roi_radius_factor=2.0,
    center=((E.shape[1] - 1)/2, (E.shape[0] - 1)/2),
    patch_radius=4,
    intensity_threshold=1e-4,
    use_normalized_intensity=True,
    local_stat="max",
    n_jobs=4,   # start with 4; later try -1
    show_progress=True,
    verbose=True,
)

Starting legacy-compatible propagation...
Field shape: (512, 512)
L = 16
wavelength = 1
nz = 50
dz = 0.30612244897959184


forward propagation:   0%|          | 0/50 [00:00<?, ?it/s]

backward propagation:   0%|          | 0/50 [00:00<?, ?it/s]

Extracting forward singularities...


[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   5 tasks      | elapsed:    4.5s
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:    5.8s
[Parallel(n_jobs=4)]: Done  17 tasks      | elapsed:    7.7s
[Parallel(n_jobs=4)]: Done  24 tasks      | elapsed:   11.1s
[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed:   14.7s
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:   19.8s
[Parallel(n_jobs=4)]: Done  51 out of  51 | elapsed:   26.5s finished


Extracting backward singularities...


[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   5 tasks      | elapsed:    4.4s
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:   19.2s
[Parallel(n_jobs=4)]: Done  17 tasks      | elapsed:   33.8s
[Parallel(n_jobs=4)]: Done  24 tasks      | elapsed:   47.5s
[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed:  1.2min
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:  1.7min
[Parallel(n_jobs=4)]: Done  50 out of  50 | elapsed:  2.2min finished


Ordering points...
Done.
Total points found: 604


# **Plan #2:** Adding a bounding Box

In [5]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import plotly.express as px
import plotly.graph_objects as go


##############################################################################################
# Propagation operators
##############################################################################################

def propTF(u1, L, la, z):
    """
    Fresnel propagation using transfer function approach.
    This version preserves your original behavior.
    """
    M, nn = u1.shape
    dx = L / M

    fx = np.arange(-1 / (2 * dx), 1 / (2 * dx), 1 / L)
    Fx, Fy = np.meshgrid(fx, fx)

    H = np.exp(-1j * np.pi * 0.25 * la * z * (Fx**2 + Fy**2))
    U2 = H * np.fft.fftshift(np.fft.fft2(u1))
    u2 = np.fft.ifft2(np.fft.ifftshift(U2))
    return u2


##############################################################################################
# Basic ordering function
##############################################################################################

def SortPoints(A0):
    AR = np.array(A0, dtype=float, copy=True)
    col, row = AR.shape

    point = AR[:, 0]
    Ord = np.zeros((col, row - 2))

    Ocol, Orow = Ord.shape
    Ord[:, -1] = point

    for ii in range(Orow - 1):
        remove1 = np.argmin(np.abs(np.sum(np.transpose(AR) - np.tile(point, (row, 1)), axis=1)))

        Ord[:, ii] = point

        AR = np.squeeze(AR[:, np.where(np.sum(np.transpose(AR) - np.tile(point, (row, 1)), axis=1) != 0)])
        col, row = AR.shape
        kk = np.argmin(np.sum((np.transpose(AR) - np.tile(point, (row, 1)))**2, axis=1))

        point = AR[:, kk]

    return Ord


##############################################################################################
# Plot of the Points
##############################################################################################

def KnotPlot(Ord):
    fig1 = px.scatter_3d(x=Ord[0, :], y=Ord[1, :], z=Ord[2, :])
    fig1.update_traces(marker=dict(color='red', size=5))

    fig4 = px.line_3d(x=Ord[0, :], y=Ord[1, :], z=Ord[2, :])
    fig4.update_traces(line=dict(color='black', width=3))

    fig3 = go.Figure(data=fig1.data + fig4.data)
    fig3.update_layout(
        autosize=False,
        width=800,
        height=800,
    )
    fig3.show()


##############################################################################################
# Intersection utilities -- preserves old behavior
##############################################################################################

def find_intersections_old(A, B):
    """
    Original-style polyline intersection finder.
    Keeps the same behavior as your old code.
    """
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)

    if len(A) < 2 or len(B) < 2:
        return np.array([]), np.array([])

    amin = lambda x1, x2: np.where(x1 < x2, x1, x2)
    amax = lambda x1, x2: np.where(x1 > x2, x1, x2)
    aall = lambda abools: np.dstack(abools).all(axis=2)

    def slope(line):
        d = np.diff(line, axis=0)
        with np.errstate(divide='ignore', invalid='ignore'):
            return d[:, 1] / d[:, 0]

    x11, x21 = np.meshgrid(A[:-1, 0], B[:-1, 0])
    x12, x22 = np.meshgrid(A[1:, 0],  B[1:, 0])
    y11, y21 = np.meshgrid(A[:-1, 1], B[:-1, 1])
    y12, y22 = np.meshgrid(A[1:, 1],  B[1:, 1])

    m1, m2 = np.meshgrid(slope(A), slope(B))

    with np.errstate(divide='ignore', invalid='ignore'):
        m1inv = 1 / m1
        m2inv = 1 / m2
        yi = (m1 * (x21 - x11 - m2inv * y21) + y11) / (1 - m1 * m2inv)
        xi = (yi - y21) * m2inv + x21

    xconds = (
        amin(x11, x12) < xi, xi <= amax(x11, x12),
        amin(x21, x22) < xi, xi <= amax(x21, x22)
    )
    yconds = (
        amin(y11, y12) < yi, yi <= amax(y11, y12),
        amin(y21, y22) < yi, yi <= amax(y21, y22)
    )

    mask = aall(xconds) & aall(yconds) & np.isfinite(xi) & np.isfinite(yi)
    return xi[mask], yi[mask]

##############################################################################################
# Bouding Box helper
##############################################################################################

def _bbox_overlap(seg1, seg2):
    """
    Fast axis-aligned bounding-box overlap test for two polylines.
    If boxes do not overlap, the curves cannot intersect.
    """
    seg1 = np.asarray(seg1, dtype=float)
    seg2 = np.asarray(seg2, dtype=float)

    x1min, y1min = np.min(seg1, axis=0)
    x1max, y1max = np.max(seg1, axis=0)

    x2min, y2min = np.min(seg2, axis=0)
    x2max, y2max = np.max(seg2, axis=0)

    return not (
        (x1max < x2min) or (x2max < x1min) or
        (y1max < y2min) or (y2max < y1min)
    )



##############################################################################################
# Contour extraction
##############################################################################################

def _contour_segments_zero(Z):
    """
    Old behavior: contours in pixel/index coordinates.
    """
    fig, ax = plt.subplots()
    cs = ax.contour(Z, levels=[0])
    plt.close(fig)

    if len(cs.allsegs) == 0:
        return []
    return cs.allsegs[0]


##############################################################################################
# Singularity extraction
##############################################################################################

# --------------------------------------------------------------------------------------------
# Single Plane Singularity extractor
# --------------------------------------------------------------------------------------------

from joblib import Parallel, delayed

def _extract_singularities_one_plane(
    U,
    ii,
    forward=True,
    JJ=None,
    use_gaussian_filter=False,
    waist_pixels=None,
    roi_radius_factor=2.0,
    center=None,
    patch_radius=3,
    intensity_threshold=1e-4,
    use_normalized_intensity=True,
    local_stat="max",
    global_intensity_max=None,
):
    Ny, Nx = U.shape

    if center is None:
        xc = (Nx - 1) / 2
        yc = (Ny - 1) / 2
    else:
        xc, yc = center

    # intensity for this plane
    Iplane = np.abs(U) ** 2
    if use_normalized_intensity and global_intensity_max is not None and global_intensity_max > 0:
        Iplane = Iplane / global_intensity_max

    do_filter = use_gaussian_filter and (waist_pixels is not None)
    roi_radius = roi_radius_factor * waist_pixels if do_filter else None

    if JJ is None:
        real_segs = _contour_segments_zero(np.real(U))
        imag_segs = _contour_segments_zero(np.imag(U))
    else:
        real_segs = _contour_segments_zero(np.real(U + JJ))
        imag_segs = _contour_segments_zero(np.imag(U + JJ))

    xi_keep = []
    yi_keep = []

    for seg1 in real_segs:
        for seg2 in imag_segs:
            # Fast rejection using axis-aligned bounding boxes
            if not _bbox_overlap(seg1, seg2):
                continue

            xinter, yinter = find_intersections_old(seg1, seg2)

            if xinter.size == 0:
                continue

            if not do_filter:
                xi_keep.extend(xinter.tolist())
                yi_keep.extend(yinter.tolist())
                continue

            for x0, y0 in zip(xinter, yinter):
                r = np.sqrt((x0 - xc) ** 2 + (y0 - yc) ** 2)
                outside_roi = r > roi_radius

                ix = int(round(x0))
                iy = int(round(y0))

                x1 = max(0, ix - patch_radius)
                x2 = min(Nx, ix + patch_radius + 1)
                y1 = max(0, iy - patch_radius)
                y2 = min(Ny, iy + patch_radius + 1)

                patch = Iplane[y1:y2, x1:x2]

                if patch.size == 0:
                    local_intensity = 0.0
                elif local_stat == "max":
                    local_intensity = np.max(patch)
                elif local_stat == "mean":
                    local_intensity = np.mean(patch)
                else:
                    raise ValueError("local_stat must be 'max' or 'mean'")

                # Drop only if BOTH are true:
                # outside ROI AND local intensity very small
                if not (outside_roi and (local_intensity < intensity_threshold)):
                    xi_keep.append(x0)
                    yi_keep.append(y0)

    xi_keep = np.asarray(xi_keep, dtype=float)
    yi_keep = np.asarray(yi_keep, dtype=float)

    sign = 1 if forward else -1
    zi_keep = sign * ii * np.ones_like(xi_keep)

    return xi_keep, yi_keep, zi_keep

# --------------------------------------------------------------------------------------------
# Parallel singularity extractor
# --------------------------------------------------------------------------------------------

def _singular_from_stack_parallel(
    field_stack,
    forward=True,
    apply_boundary_mask=False,
    boundary_radius=70,
    use_gaussian_filter=False,
    waist_pixels=None,
    roi_radius_factor=2.0,
    center=None,
    patch_radius=3,
    intensity_threshold=1e-4,
    use_normalized_intensity=True,
    local_stat="max",
    n_jobs=-1,
    verbose=True,
):
    Fstack = np.asarray(field_stack)
    if Fstack.ndim != 3:
        raise ValueError("field_stack must have shape (Nz, Ny, Nx)")

    Nz, Ny, Nx = Fstack.shape

    if center is None:
        xc = (Nx - 1) / 2
        yc = (Ny - 1) / 2
    else:
        xc, yc = center

    if apply_boundary_mask:
        xx = np.arange(Nx)
        yy = np.arange(Ny)
        xf, yf = np.meshgrid(xx, yy)
        JJ = np.sqrt((xf - xc)**2 + (yf - yc)**2) > boundary_radius
    else:
        JJ = None

    global_intensity_max = np.max(np.abs(Fstack) ** 2) if use_normalized_intensity else None

    results = Parallel(n_jobs=n_jobs, prefer="processes", verbose=10 if verbose else 0)(
        delayed(_extract_singularities_one_plane)(
            U=Fstack[ii],
            ii=ii,
            forward=forward,
            JJ=JJ,
            use_gaussian_filter=use_gaussian_filter,
            waist_pixels=waist_pixels,
            roi_radius_factor=roi_radius_factor,
            center=center,
            patch_radius=patch_radius,
            intensity_threshold=intensity_threshold,
            use_normalized_intensity=use_normalized_intensity,
            local_stat=local_stat,
            global_intensity_max=global_intensity_max,
        )
        for ii in range(Nz)
    )

    X_list, Y_list, Z_list = [], [], []

    for xk, yk, zk in results:
        if xk.size:
            X_list.append(xk)
            Y_list.append(yk)
            Z_list.append(zk)

    if not X_list:
        return np.zeros((3, 0), dtype=float)

    return np.array([
        np.concatenate(X_list),
        np.concatenate(Y_list),
        np.concatenate(Z_list),
    ], dtype=float)



##############################################################################################
# Main propagation + knot extraction
##############################################################################################

def compute_knot_propagation_legacy_compatible(
    E0,
    L,
    wavelength=1.0,
    Z0=15.0,
    nz=50,
    do_backward=True,
    order_points=True,
    apply_forward_mask=True,
    boundary_radius=70,
    use_gaussian_filter=False,
    waist_pixels=None,
    roi_radius_factor=2.0,
    center=None,
    patch_radius=3,
    intensity_threshold=1e-4,
    use_normalized_intensity=True,
    local_stat="max",
    n_jobs=-1,
    show_progress=True,
    verbose=True,
):
    """
    Version that reproduces the behavior of your old code as closely as possible,
    with optional Gaussian-based filtering and parallel singularity extraction.

    Parameters
    ----------
    E0 : 2D complex array
        Input field.
    L : float
        Window size passed into propTF, equivalent to your old 2*maxx.
    wavelength : float
        Wavelength parameter passed to propTF.
    Z0 : float
        Total propagation distance.
    nz : int
        Number of propagation steps.
    """

    E0 = np.asarray(E0, dtype=complex)

    if E0.ndim != 2:
        raise ValueError("E0 must be a 2D field.")
    if nz <= 0:
        raise ValueError("nz must be > 0.")
    if wavelength <= 0:
        raise ValueError("wavelength must be > 0.")

    Z = np.linspace(0, Z0, nz)
    dz = np.abs(Z[0] - Z[1]) if nz > 1 else Z0

    if verbose:
        print("Starting legacy-compatible propagation...")
        print(f"Field shape: {E0.shape}")
        print(f"L = {L}")
        print(f"wavelength = {wavelength}")
        print(f"nz = {nz}")
        print(f"dz = {dz}")

    # ------------------------------------------------------------------
    # Forward propagation: old behavior
    # ------------------------------------------------------------------
    F = [E0.copy()]
    U = E0.copy()

    iterator = range(nz)
    if show_progress:
        iterator = tqdm(iterator, total=nz, desc="forward propagation", leave=True)

    for _ in iterator:
        U = propTF(U, L, wavelength, dz)
        F.append(U.copy())

    # ------------------------------------------------------------------
    # Backward propagation: old behavior
    # ------------------------------------------------------------------
    FB = []
    if do_backward:
        U = E0.copy()

        iterator = range(1, nz + 1)
        if show_progress:
            iterator = tqdm(iterator, total=nz, desc="backward propagation", leave=True)

        for _ in iterator:
            U = propTF(U, L, wavelength, -dz)
            FB.append(U.copy())

    # ------------------------------------------------------------------
    # Singularity extraction
    # ------------------------------------------------------------------
    if verbose:
        print("Extracting forward singularities...")

    H = _singular_from_stack_parallel(
        F,
        forward=True,
        apply_boundary_mask=apply_forward_mask,
        boundary_radius=boundary_radius,
        use_gaussian_filter=use_gaussian_filter,
        waist_pixels=waist_pixels,
        roi_radius_factor=roi_radius_factor,
        center=center,
        patch_radius=patch_radius,
        intensity_threshold=intensity_threshold,
        use_normalized_intensity=use_normalized_intensity,
        local_stat=local_stat,
        n_jobs=n_jobs,
        verbose=verbose,
    )

    if do_backward:
        if verbose:
            print("Extracting backward singularities...")

        H2 = _singular_from_stack_parallel(
            FB,
            forward=False,
            apply_boundary_mask=False,
            boundary_radius=boundary_radius,
            use_gaussian_filter=use_gaussian_filter,
            waist_pixels=waist_pixels,
            roi_radius_factor=roi_radius_factor,
            center=center,
            patch_radius=patch_radius,
            intensity_threshold=intensity_threshold,
            use_normalized_intensity=use_normalized_intensity,
            local_stat=local_stat,
            n_jobs=n_jobs,
            verbose=verbose,
        )
    else:
        H2 = np.zeros((3, 0), dtype=float)

    # ------------------------------------------------------------------
    # Combine and order
    # ------------------------------------------------------------------
    TotalPoints = np.concatenate((H, H2), axis=1)

    if order_points and TotalPoints.shape[1] > 2:
        if verbose:
            print("Ordering points...")
        ord_points = SortPoints(TotalPoints)
    else:
        ord_points = TotalPoints

    if verbose:
        print("Done.")
        print(f"Total points found: {TotalPoints.shape[1]}")

    return {
        "E0": E0,
        "L": L,
        "wavelength": wavelength,
        "Z0": Z0,
        "nz": nz,
        "dz": dz,
        "forward_fields": np.asarray(F, dtype=complex),
        "backward_fields": np.asarray(FB, dtype=complex) if do_backward else None,
        "singular_forward": H,
        "singular_backward": H2,
        "total_points": TotalPoints,
        "ordered_points": ord_points,
    }

In [6]:
# Creation of the Knot
knot = OpticalKnot(s=2, r=3, ell=1, a=1, b=1)

maxx=8
NN=512
w0_t = 1.6

A=knot.make_field(N=NN, xmax=maxx, w0=w0_t)

E=A[4]
dx = (2*maxx) / E.shape[0]
w0_pix = int(w0_t/ dx)

result = compute_knot_propagation_legacy_compatible(
    E0=E,
    L=2*maxx,
    wavelength=1,
    Z0=15,
    nz=50,
    do_backward=True,
    order_points=True,
    apply_forward_mask=True,
    boundary_radius=70,
    use_gaussian_filter=True,
    waist_pixels=51.2,
    roi_radius_factor=2.0,
    center=((E.shape[1] - 1)/2, (E.shape[0] - 1)/2),
    patch_radius=4,
    intensity_threshold=1e-4,
    use_normalized_intensity=True,
    local_stat="max",
    n_jobs=4,   # start with 4; later try -1
    show_progress=True,
    verbose=True,
)

Starting legacy-compatible propagation...
Field shape: (512, 512)
L = 16
wavelength = 1
nz = 50
dz = 0.30612244897959184


forward propagation:   0%|          | 0/50 [00:00<?, ?it/s]

backward propagation:   0%|          | 0/50 [00:00<?, ?it/s]

Extracting forward singularities...


[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   5 tasks      | elapsed:    5.6s
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:    6.4s
[Parallel(n_jobs=4)]: Done  17 tasks      | elapsed:    7.3s
[Parallel(n_jobs=4)]: Done  24 tasks      | elapsed:    9.7s
[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed:   13.5s
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:   15.7s
[Parallel(n_jobs=4)]: Done  51 out of  51 | elapsed:   18.2s finished


Extracting backward singularities...


[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   5 tasks      | elapsed:    1.3s
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:    5.5s
[Parallel(n_jobs=4)]: Done  17 tasks      | elapsed:   13.0s
[Parallel(n_jobs=4)]: Done  24 tasks      | elapsed:   22.3s
[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed:   42.6s
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:  1.2min
[Parallel(n_jobs=4)]: Done  50 out of  50 | elapsed:  1.6min finished


Ordering points...
Done.
Total points found: 604
